In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, cross_val_predict, cross_validate
from sklearn.linear_model import LinearRegression
from scipy.stats import gaussian_kde
from scipy.spatial import cKDTree
import shap 

sys.path.insert(1, '../../ProcessEvents')
from config import CATCHMENT_LOOKUP_DICT, ENSEMBLE_MEMBERS, CATCHMENTS

from PrepDataFunctions import *
from ModellingFunctions import groups_hierarchical

### Get event data

In [6]:
df = prep_for_modelling(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/all_catchments_with_topo_and_temporal.pkl", 
                        remove_events_with_zero_flooding = False)
df.reset_index(inplace=True, drop=True)
# df[df['max_quartile_index_n'].isnull()][["max_precip", "max_quartile_index_n", "length", "total_acc", "length_of_active_rain"]]
# df = create_subset(df, 10000, 10)
subset_early = create_temporal_subset(df, "before", 2020)

### Create model instances and parameters

In [4]:
rf = RandomForestRegressor(n_estimators=100,max_depth=8,random_state=42)
lr = LinearRegression()
cv_parameters = KFold(n_splits=5, shuffle=True, random_state=42)
error_metrics = ['neg_mean_absolute_error', 'r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_percentage_error']

### Define parameters to use in modelling

In [10]:
my_vars = ['max_precip', 'total_acc',  
                       "hc_at_peak", 'fumax_at_peak', 
                      'se_max', 'se_min',  'sat_at_peak_new', 'sat_at_peak_old',
                      'fu_mean_m',  'fu_min_m', 'fu_max_m', 'fu_at_peak_new', 
                        "mean_sm_2_before_event",
                      #   'peak_position_ratio','d50','gini','time_based_std','dry_ratio','max_quartile_index',
         'peak_position_ratio_n','d50_n','gini_n','time_based_std_n','dry_ratio_n','max_quartile_index_n',                     
                      'peak_cell_slope_avg', 'peak_cell_elev_avg','peak_cell_elev_min','peak_cell_slope_max', 
                      #   'peak_cell_elev_max',
                      'peak_cell_urban_prop', 'peak_cell_inside_prop',
                   # take these out because highly correlated with max_precip    
                   #"cluster_mean_intensity_t80","cluster_mean_intensity_t50","cluster_mean_intensity_t60",
           # "cluster_mean_intensity_t50", "cluster_mean_intensity_t60", "cluster_mean_intensity_t80",
                #'cluster_rain_total_t50',  'cluster_rain_total_t60','cluster_rain_total_t80', 
           'cluster_n_cells_t50', 'cluster_n_cells_t80',
                      "random_variable"]

# Define X (predictors) and Y (response)
X = subset_early[my_vars]
y = subset_early['10cm_area']

### Fit model 

In [14]:
rf.fit(X,y)

### Save model

In [16]:
import pickle
filename = '/scratch/hydro4/users/kv25483/FutureFlood/Data/Models/rf_model.pkl'
pickle.dump(rf, open(filename, 'wb'))